<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 2</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">2.5</span>Reporte automatizado y dashboard</div>
</div>

**Objetivo.** generar un informe HTML completo con una sola ejecución, y un dashboard sencillo con selector.

**Lo que vamos a hacer**

- Calcular métricas de varios activos.
- Armar un informe HTML autocontenido (tabla + gráficos).
- Crear un dashboard con selector de activo.
- Ver cómo se vuelve un script que corre solo (`report.py`).

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Datos y métricas

In [ ]:
import base64, io, pathlib
from datetime import date
from IPython.display import IFrame

tickers = ["EC", "CIB", "AAPL", "MSFT", "GLD"]
prices = download_prices(tickers, start="2022-01-01")
rets = to_returns(prices, log=False)
out = pathlib.Path("output"); out.mkdir(exist_ok=True)

def fig_b64(fig):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

> ### ✏️ Ejercicio 1 — Tabla de métricas
> Completa `build_stats(rets)`: retorno anual, volatilidad, Sharpe, **VaR 95% diario** y **máximo drawdown** por activo.

In [ ]:
def build_stats(rets):
    return pd.DataFrame({
        "Retorno anual": annualized_return(rets),
        "Volatilidad": annualized_vol(rets),
        "Sharpe": sharpe_ratio(rets),
        # TODO: añade "VaR 95% (1d)" y "Máx. drawdown"
    })

stats = build_stats(rets)
stats

In [ ]:
def build_stats(rets):
    return pd.DataFrame({
        "Retorno anual": annualized_return(rets),
        "Volatilidad": annualized_vol(rets),
        "Sharpe": sharpe_ratio(rets),
        "VaR 95% (1d)": -rets.quantile(0.05),
        "Máx. drawdown": (1 + rets).cumprod().apply(max_drawdown),
    })

stats = build_stats(rets)
stats

## 2. El informe

In [ ]:
CSS = """<style>body{font-family:Georgia,serif;max-width:900px;margin:2rem auto;padding:0 1rem;color:#1f1f1f}
h1{font-family:'Arial Narrow',Arial,sans-serif;text-transform:uppercase;font-weight:300;border-top:10px solid #A6192E;padding-top:14px}
table{border-collapse:collapse;width:100%}th,td{padding:6px 10px;border-bottom:1px solid #d9d9d9;text-align:right}
th:first-child,td:first-child{text-align:left}th{border-top:2px solid #1f1f1f;border-bottom:1px solid #1f1f1f}img{max-width:100%}</style>"""

def build_report(stats, prices):
    fig, ax = plt.subplots()
    (prices / prices.iloc[0] * 100).plot(ax=ax, title="Evolución (base 100)")
    img = fig_b64(fig)
    fmt = {c: "{:.1%}".format for c in stats.columns if c != "Sharpe"}; fmt["Sharpe"] = "{:.2f}".format
    return f"""<!doctype html><html lang="es"><meta charset="utf-8"><title>Informe</title>{CSS}
<h1>Informe de mercado</h1><p>{date.today():%d/%m/%Y} · {", ".join(stats.index)}</p>
<h2>Métricas</h2>{stats.to_html(formatters=fmt, border=0)}
<h2>Evolución</h2><img src="data:image/png;base64,{img}"></html>"""

> ### ✏️ Ejercicio 2 — Guardar y ver el informe
> Genera el HTML y guárdalo como `output/informe_AAAA-MM.html` (usa la fecha de hoy). Luego muéstralo con `IFrame`.

In [ ]:
# TODO: html = ...; guardar en `path`; mostrar
path = out / "informe.html"

In [ ]:
html = build_report(stats, prices)
path = out / f"informe_{date.today():%Y-%m}.html"
path.write_text(html, encoding="utf-8")
print("Guardado en", path)
IFrame(str(path), width="100%", height=520)

## 3. Dashboard con selector

In [ ]:
panels = {}
for t in prices.columns:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    prices[t].plot(ax=ax[0], title=f"{t} — precio")
    rets[t].hist(bins=40, ax=ax[1], color="#d9d9d9", edgecolor="white"); ax[1].set_title("Retornos diarios")
    panels[t] = fig_b64(fig)

options = "".join(f'<option value="{t}">{t}</option>' for t in panels)
imgs = "".join(f'<img id="p_{t}" src="data:image/png;base64,{b}" style="display:{"block" if i == 0 else "none"}">' for i, (t, b) in enumerate(panels.items()))
dash = f"""<!doctype html><html lang="es"><meta charset="utf-8"><title>Dashboard</title>{CSS}
<h1>Dashboard</h1><select id="s" onchange="show(this.value)">{options}</select>
<div id="stats">{stats.round(3).to_html(border=0)}</div>{imgs}
<script>function show(t){{document.querySelectorAll("img").forEach(i=>i.style.display="none");document.getElementById("p_"+t).style.display="block"}}</script></html>"""
(out / "dashboard.html").write_text(dash, encoding="utf-8")
IFrame("output/dashboard.html", width="100%", height=520)

**Del notebook al script.** Todo esto vive también en `report.py` (carpeta de esta sesión). Se ejecuta desde la terminal y puede programarse con `cron` o GitHub Actions:

```bash
python report.py EC CIB AAPL --start 2022-01-01 --out output/informe.html
```

## Para llevarse

- Un informe es una función: cambia la lista de activos o el mes y se regenera completo.
- HTML autocontenido (gráficos en base64) se envía por correo sin archivos adjuntos.
- Programar el script lo convierte en un reporte que se genera solo.

---
*Excel para mirar, Python para repetir.*